[← Lekcja 1: Położenie](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_1_polozenie.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 2 z 2** (następne wkrótce)

# 🐝 Kinematyka – część 2: Dokąd i jak szybko leci pszczoła? (prędkość)

W części 1 opisaliśmy **położenie** pszczoły wektorem $\vec{r}$. Ale pszczoły latają! Gdy pszczoła się porusza, jej wektor położenia się zmienia. W tym zeszycie:

1. opiszemy, **jak bardzo** zmieniło się położenie (przemieszczenie),
2. zdefiniujemy **prędkość** i zobaczymy, że jest **wektorem**,
3. poprowadzimy pszczołę do kwiatka, dobierając odpowiednie parametry lotu. 🌼

Zacznij od uruchomienia kodu pomocniczego (rysowanie i animacja):

In [ ]:
#@title 🔧 Kod pomocniczy – rysowanie (uruchom i nie przejmuj się nim, wrócimy do niego później)
import numpy as np
import plotly.graph_objects as go

R_PSZCZOLY = 0.25   # promień pszczoły [m]

# kamera: widok domyślny i widoki z różnych stron
WIDOKI = {
    "↺ Reset widoku": dict(eye=dict(x=1.6, y=-1.6, z=1.0), up=dict(x=0, y=0, z=1)),
    "Z góry":        dict(eye=dict(x=0.0, y=0.001, z=2.6), up=dict(x=0, y=1, z=0)),
    "Z przodu":      dict(eye=dict(x=0.0, y=-2.6, z=0.0), up=dict(x=0, y=0, z=1)),
    "Z boku":        dict(eye=dict(x=2.6, y=0.0, z=0.0), up=dict(x=0, y=0, z=1)),
}

def pszczola(p, R=R_PSZCZOLY):
    """Kula w żółto-czarne paski ze środkiem w punkcie p."""
    x, y, z = p
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 40))
    return go.Surface(
        x=x + R*np.cos(u)*np.sin(v), y=y + R*np.sin(u)*np.sin(v), z=z + R*np.cos(v),
        surfacecolor=(np.floor(v/(np.pi/7)) % 2),
        colorscale=[[0, "#FFD400"], [0.5, "#FFD400"], [0.5, "#1a1a1a"], [1, "#1a1a1a"]],
        cmin=0, cmax=1, showscale=False, hoverinfo="skip",
        lighting=dict(ambient=0.9, diffuse=0.3, specular=0.05),
    )

def strzalka(poczatek, w, kolor, dl_grotu=0.6, r_grotu=0.18):
    """Strzałka-wektor: trzon (linia) + grot (stożek z trójkątów). Czubek leży w punkcie poczatek + w."""
    poczatek = np.asarray(poczatek, float)
    w = np.asarray(w, float)
    d = np.linalg.norm(w)
    e = w / d if d > 1e-9 else np.array([0, 0, 1.0])
    dl = min(dl_grotu, d)                          # krótkie wektory dostają mniejszy grot
    r = r_grotu * dl / dl_grotu
    ostrze = poczatek + w
    podstawa = ostrze - e * dl
    trzon = go.Scatter3d(x=[poczatek[0], podstawa[0]], y=[poczatek[1], podstawa[1]],
                         z=[poczatek[2], podstawa[2]], mode="lines",
                         line=dict(color=kolor, width=7), hoverinfo="skip")
    a = np.cross(e, [1, 0, 0]) if abs(e[0]) < 0.9 else np.cross(e, [0, 1, 0])
    a = a / np.linalg.norm(a)
    b = np.cross(e, a)
    t = np.linspace(0, 2*np.pi, 16, endpoint=False)
    kolo = podstawa + r*(np.outer(np.cos(t), a) + np.outer(np.sin(t), b))
    punkty = np.vstack([ostrze, kolo, podstawa])   # 0 = czubek, 1..16 = okrąg, 17 = środek podstawy
    n = len(t)
    boki = [(0, 1 + k, 1 + (k + 1) % n) for k in range(n)]
    spod = [(n + 1, 1 + (k + 1) % n, 1 + k) for k in range(n)]
    ij = np.array(boki + spod)
    grot = go.Mesh3d(x=punkty[:, 0], y=punkty[:, 1], z=punkty[:, 2],
                     i=ij[:, 0], j=ij[:, 1], k=ij[:, 2], color=kolor, flatshading=True,
                     lighting=dict(ambient=0.8, diffuse=0.4), hoverinfo="skip")
    return [trzon, grot]

def osie_ukladu(L):
    return go.Scatter3d(x=[-L, L, None, 0, 0, None, 0, 0], y=[0, 0, None, -L, L, None, 0, 0],
                        z=[0, 0, None, 0, 0, None, -L, L], mode="lines",
                        line=dict(color="gray", width=3), hoverinfo="skip")

def menu_widokow():
    return dict(type="buttons", direction="right", x=0.0, xanchor="left", y=1.0, yanchor="bottom",
                buttons=[dict(label=n, method="relayout", args=[{"scene.camera": k}])
                         for n, k in WIDOKI.items()])

def menu_ukrywania(opcje, ile, adnotacje=None):
    """Lista rozwijana 'co pokazać'. opcje = [(napis, [indeksy do ukrycia], czy_ukryc_etykiete), ...]"""
    przyciski = []
    for napis, ukryte, ukryj_etykiete in opcje:
        widoczne = [i not in ukryte for i in range(ile)]
        zmiana_ukladu = {} if adnotacje is None else {"scene.annotations": [] if ukryj_etykiete else adnotacje}
        przyciski.append(dict(label=napis, method="update", args=[{"visible": widoczne}, zmiana_ukladu]))
    return dict(type="dropdown", direction="down", x=1.0, xanchor="right", y=1.0, yanchor="bottom",
                showactive=True, buttons=przyciski)

def zrob_uklad(tytul, zakres, dtick, adnotacje=(), menu=(), sliders=(), gora=70, dol=0):
    def os_(nazwa):
        return dict(title=nazwa, range=[-zakres, zakres], dtick=dtick)
    return go.Layout(
        title=dict(text=tytul, x=0.5, y=0.98, yanchor="top"),
        scene=dict(xaxis=os_("x [m]"), yaxis=os_("y [m]"), zaxis=os_("z [m]"), aspectmode="cube",
                   camera=WIDOKI["↺ Reset widoku"], annotations=list(adnotacje)),
        height=680, margin=dict(l=0, r=0, t=gora, b=dol), showlegend=False,
        updatemenus=[menu_widokow()] + list(menu), sliders=list(sliders),
        uirevision="stala",   # zachowuje obrót wykresu podczas animacji i zmian
    )

L = 6   # osie rysujemy od -L do L

def elipsoida(srodek, a, b, c, fi):
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 20), np.linspace(0, np.pi, 14))
    px, py, pz = a*np.cos(u)*np.sin(v), b*np.sin(u)*np.sin(v), c*np.cos(v)
    return (srodek[0] + px*np.cos(fi) - py*np.sin(fi),
            srodek[1] + px*np.sin(fi) + py*np.cos(fi),
            srodek[2] + pz)

def kwiatek_slady(c):
    """Kwiatek: żółty środek + 8 białych płatków (bez łodygi)."""
    c = np.asarray(c, float)
    x, y, z = elipsoida(c, 0.3, 0.3, 0.3, 0)
    slady = [go.Surface(x=x, y=y, z=z, surfacecolor=np.zeros_like(x), cmin=0, cmax=1, showscale=False,
                        colorscale=[[0, "#FFE000"], [1, "#FFE000"]], hoverinfo="skip",
                        lighting=dict(ambient=0.9, diffuse=0.3))]
    ox, oy, oz = [], [], []
    t = np.linspace(0, 2*np.pi, 30)
    for fi in np.linspace(0, 2*np.pi, 8, endpoint=False):
        srodek = c + 0.55*np.array([np.cos(fi), np.sin(fi), 0])
        x, y, z = elipsoida(srodek, 0.32, 0.16, 0.07, fi)
        slady.append(go.Surface(x=x, y=y, z=z, surfacecolor=np.zeros_like(x), cmin=0, cmax=1,
                                showscale=False, colorscale=[[0, "white"], [1, "white"]],
                                lighting=dict(ambient=0.75, diffuse=0.5), hoverinfo="skip"))
        px, py = 0.32*np.cos(t), 0.16*np.sin(t)       # obrys płatka (żeby biel było widać)
        ox += list(srodek[0] + px*np.cos(fi) - py*np.sin(fi)) + [None]
        oy += list(srodek[1] + px*np.sin(fi) + py*np.cos(fi)) + [None]
        oz += [srodek[2]]*len(t) + [None]
    slady.append(go.Scatter3d(x=ox, y=oy, z=oz, mode="lines", line=dict(color="gray", width=2),
                              hoverinfo="skip"))
    return slady

def lot(r1, v, dt, kwiatek=None):
    """Animacja lotu pszczoły: start w r1, stała prędkość v, czas lotu dt."""
    r1 = np.asarray(r1, float)
    v = np.asarray(v, float)
    if dt <= 0:
        print("⚠️ Czas lotu dt musi być większy od zera.")
        return
    r2 = r1 + v*dt
    N = 40                                              # liczba klatek animacji
    czasy = np.linspace(0, dt, N + 1)

    def ruchome(p):   # 0 pszczoła | 1-2 wektor r (czerwony) | 3-4 wektor v (niebieski)
        return [pszczola(p)] + strzalka([0, 0, 0], p, "red") + strzalka(p, v, "royalblue")

    stale = [
        go.Scatter3d(x=[r1[0], r2[0]], y=[r1[1], r2[1]], z=[r1[2], r2[2]], mode="lines",
                     line=dict(color="gray", width=3, dash="dash"), hoverinfo="skip"),      # 5 tor
        go.Scatter3d(x=[r1[0], r2[0]], y=[r1[1], r2[1]], z=[r1[2], r2[2]], mode="markers+text",
                     marker=dict(size=5, color="gray"), text=["start", "koniec lotu"],
                     textposition="top center", hoverinfo="skip"),                          # 6 znaczniki
    ]
    if kwiatek is not None:
        stale += kwiatek_slady(kwiatek)
    stale.append(osie_ukladu(L))
    dane = ruchome(r1) + stale
    ramki = [go.Frame(data=ruchome(r1 + v*t), traces=[0, 1, 2, 3, 4], name=str(k))
             for k, t in enumerate(czasy)]

    czas_klatki = int(1000*min(dt, 8)/N)
    suwak = dict(active=0, x=0.12, len=0.86, xanchor="left", y=0, yanchor="top",
                 pad=dict(t=10, b=10), font=dict(color="rgba(0,0,0,0)"),
                 currentvalue=dict(prefix="t = ", suffix=" s", font=dict(size=16)),
                 steps=[dict(method="animate", label=f"{t:.1f}",
                             args=[[str(k)], dict(mode="immediate",
                                                  frame=dict(duration=0, redraw=True),
                                                  transition=dict(duration=0))])
                        for k, t in enumerate(czasy)])
    odtwarzanie = dict(type="buttons", direction="left", x=0.0, xanchor="left", y=0, yanchor="top",
                       pad=dict(t=10), showactive=False, buttons=[
        dict(label="▶ Start", method="animate",
             args=[None, dict(frame=dict(duration=czas_klatki, redraw=True), fromcurrent=True,
                              transition=dict(duration=0))]),
        dict(label="⏸ Pauza", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate",
                                transition=dict(duration=0))])])
    lista = menu_ukrywania([("Pokaż wszystko", [], False), ("Ukryj pszczołę", [0], False),
                            ("Ukryj wektor r", [1, 2], False), ("Ukryj wektor v", [3, 4], False),
                            ("Ukryj tor lotu", [5, 6], False)], ile=len(dane))
    tytul = (f"<b>v</b> = ({v[0]:g}; {v[1]:g}; {v[2]:g}) m/s,   Δt = {dt:g} s"
             "<br><sup><span style=\"color:red\"><b>r</b></span> – wektor położenia,   "
             "<span style=\"color:royalblue\"><b>v</b></span> – wektor prędkości "
             "(na wykresie: 1 m/s = 1 m)</sup>")
    adnotacje = []
    if kwiatek is not None:
        adnotacje.append(dict(x=kwiatek[0], y=kwiatek[1], z=kwiatek[2] + 0.9, text="<b>kwiatek</b>",
                              showarrow=False, font=dict(size=15)))
    uklad = zrob_uklad(tytul, L, 2, adnotacje=adnotacje, menu=[lista, odtwarzanie],
                       sliders=[suwak], gora=110, dol=110)

    print(f"Prędkość: |v| = {np.linalg.norm(v):.2f} m/s   |   położenie końcowe r2 = {np.round(r2, 2)} m")
    if np.any(np.abs(r2) > L):
        print("ℹ️ Pszczoła wyleciała poza obszar wykresu – końcówka lotu nie jest widoczna.")
    if kwiatek is not None:
        odl = np.linalg.norm(r2 - np.asarray(kwiatek, float))
        if odl <= 0.5:
            print("🎉 Pszczoła dotarła do kwiatka!")
        else:
            print(f"🐝 Jeszcze nie... pszczoła kończy lot {odl:.2f} m od kwiatka.")
    go.Figure(data=dane, layout=uklad, frames=ramki).show()

## 1. Przemieszczenie

Pszczoła w chwili $t_1$ jest w punkcie $\vec{r}_1$, a w chwili $t_2$ w punkcie $\vec{r}_2$. **Zmianę położenia** nazywamy **przemieszczeniem**:

$$\Delta\vec{r} = \vec{r}_2 - \vec{r}_1$$

Grecka litera $\Delta$ (delta) czytamy jako „zmiana" i zawsze oznacza: *wartość końcowa minus wartość początkowa*. Tak samo **czas lotu** to:

$$\Delta t = t_2 - t_1$$

Przemieszczenie $\Delta\vec{r}$ jest wektorem poprowadzonym **od punktu startu do punktu końcowego**. Nie obchodzi go, czy pszczoła leciała prosto, czy zygzakiem – liczy się tylko, gdzie była na początku i gdzie jest na końcu. (Długość całej trasy to *droga* – to coś innego.)

In [ ]:
# położenie pszczoły na początku i na końcu lotu (w metrach)
r1 = np.array([-4, -2, 1])
r2 = np.array([ 0,  0, 2])

# chwile czasu (w sekundach)
t1 = 0
t2 = 2

dr = r2 - r1      # przemieszczenie
dt = t2 - t1      # czas lotu

print("Przemieszczenie Δr =", dr, "m")
print("Czas lotu Δt =", dt, "s")

> 💡 **Uwaga o Pythonie:** działania na wektorach `np.array` wykonują się **osobno na każdej składowej**. Zapis `r2 - r1` oznacza więc $(x_2 - x_1;\ y_2 - y_1;\ z_2 - z_1)$. Nie trzeba pisać pętli!

## 2. Prędkość

Im większe przemieszczenie w tym samym czasie, tym szybciej leci pszczoła. **Prędkość** to przemieszczenie przypadające na jednostkę czasu:

$$\vec{v} = \frac{\Delta\vec{r}}{\Delta t}$$

Jednostką jest **metr na sekundę** (m/s). Dzielimy wektor przez zwykłą liczbę, więc dzielimy każdą składową osobno:

$$v_x = \frac{\Delta x}{\Delta t}, \qquad v_y = \frac{\Delta y}{\Delta t}, \qquad v_z = \frac{\Delta z}{\Delta t}$$

Ponieważ $\Delta t > 0$, wektor prędkości $\vec{v}$ jest skierowany **tak samo jak przemieszczenie** $\Delta\vec{r}$: pszczoła leci w tę stronę, w którą się przemieszcza.

In [ ]:
v = dr / dt

print("Wektor prędkości v =", v, "m/s")
print("Składowa vx =", v[0], "m/s")
print("Składowa vy =", v[1], "m/s")
print("Składowa vz =", v[2], "m/s")

szybkosc = np.linalg.norm(v)    # długość wektora v
print(f"Wartość prędkości |v| = {szybkosc:.2f} m/s")

> 💡 **Wektor i jego wartość.** $\vec{v}$ (ze strzałką) to wektor: mówi, **dokąd** i **jak szybko** leci pszczoła. $|\vec{v}| = \sqrt{v_x^2 + v_y^2 + v_z^2}$ to tylko jego długość, czyli **wartość prędkości** (potocznie „szybkość"). W Pythonie liczy ją `np.linalg.norm(v)` – to ta sama operacja co pierwiastek z sumy kwadratów w części 1.

> ℹ️ **Ścisłość.** Wzór $\vec{v} = \Delta\vec{r}/\Delta t$ daje **prędkość średnią** w czasie $\Delta t$. W tej lekcji pszczoła leci ze **stałą** prędkością, więc prędkość w każdej chwili jest taka sama jak średnia.

## 3. Zobaczmy to na wykresie

Poniżej pszczoła leci z punktu $\vec{r}_1$ z prędkością $\vec{v}$ przez czas $\Delta t$ – dokładnie tak, jak obliczyliśmy.

In [ ]:
lot(r1, v, dt)

### Jak czytać animację?
- **▶ Start** puszcza lot, **⏸ Pauza** go zatrzymuje. Suwak pod wykresem pokazuje czas $t$ i pozwala „przewijać" lot.
- **Niebieski wektor** to $\vec{v}$, przyczepiony do pszczoły. **Czerwony wektor** to wektor położenia $\vec{r}$ z części 1 – rośnie i zmienia kierunek wraz z lotem.
- Długość niebieskiego wektora na wykresie to wartość prędkości w skali **1 m/s = 1 m**.
- Wykres możesz obracać, a lista w prawym górnym rogu pozwala **ukryć pszczołę, wektory lub tor lotu**.
- Szara przerywana linia to tor lotu: od punktu *start* do punktu *koniec lotu*. Leży na niej także wektor $\Delta\vec{r}$.

> ✏️ **Zadanie 1.** Obserwuj niebieski wektor w trakcie lotu. Czy jego kierunek i długość się zmieniają? Dlaczego? Porównaj jego kierunek z kierunkiem szarego toru.

> ✏️ **Zadanie 2.** Wróć do komórki z przemieszczeniem i zmień `t2 = 2` na `t2 = 4` (ta sama trasa, dwa razy więcej czasu). Uruchom ponownie tę komórkę, a potem dwie następne (z prędkością i z wykresem). Jak zmieniła się prędkość? Jak zmienił się niebieski wektor?

## 4. Przekształcamy wzór

Wzór $\vec{v} = \Delta\vec{r}/\Delta t$ możemy odwrócić. Mnożymy obie strony przez $\Delta t$:

$$\Delta\vec{r} = \vec{v}\,\Delta t$$

Ponieważ $\Delta\vec{r} = \vec{r}_2 - \vec{r}_1$, dostajemy wzór na **położenie końcowe**:

$$\vec{r}_2 = \vec{r}_1 + \vec{v}\,\Delta t$$

To bardzo użyteczne: **znając punkt startu, prędkość i czas lotu, możemy przewidzieć, gdzie pszczoła się znajdzie**.

In [ ]:
r2_przewidziane = r1 + v * dt

print("Przewidziane położenie końcowe:", r2_przewidziane, "m")
print("Czy zgadza się z r2 z początku?", np.allclose(r2_przewidziane, r2))

## 5. 🌼 Zadanie: lot do kwiatka

Pszczoła startuje w punkcie $\vec{r}_1 = (-4;\ -2;\ 1)$ m. W punkcie $(4;\ 2;\ 3)$ m rośnie kwiatek. Pszczoła poleci **ze stałą prędkością** $\vec{v}$ przez czas $\Delta t$.

**Twoje zadanie:** dobierz prędkość `vx`, `vy`, `vz` (w m/s) oraz czas lotu `dt` (w sekundach) tak, aby pszczoła znalazła się **w kwiatku** (z dokładnością do 0,5 m).

Zmieniaj liczby w sekcji między liniami i uruchamiaj komórkę. Pod wykresem dostaniesz informację, czy się udało. Animację możesz puścić przyciskiem ▶.

In [ ]:
r1 = np.array([-4, -2, 1])       # start pszczoły [m]
kwiatek = np.array([4, 2, 3])    # położenie kwiatka [m]

# ----------- ZMIENIAJ TE LICZBY -----------
vx = 1      # [m/s]
vy = 1      # [m/s]
vz = 0      # [m/s]
dt = 3      # czas lotu [s]
# ------------------------------------------

v = np.array([vx, vy, vz])
lot(r1, v, dt, kwiatek)

<details>
<summary>💡 Podpowiedź 1 (kliknij, jeśli utkniesz)</summary>

Pomyśl, jakie **przemieszczenie** $\Delta\vec{r}$ musi wykonać pszczoła, żeby z punktu startu trafić do kwiatka. Możesz je obliczyć w Pythonie: `kwiatek - r1`.
</details>

<details>
<summary>💡 Podpowiedź 2</summary>

Wybierz dowolny czas lotu (np. 4 s) i skorzystaj ze wzoru $\vec{v} = \Delta\vec{r}/\Delta t$. W Pythonie:

```python
dr_potrzebne = kwiatek - r1
dt = 4
v = dr_potrzebne / dt
print(v)
```

Wpisz otrzymane liczby jako `vx`, `vy`, `vz` w komórce wyżej.
</details>

> 🤔 **Pytania.** Czy istnieje tylko jedno rozwiązanie? Co się dzieje z wartością prędkości, gdy skracasz czas lotu? Czy pszczoła może dolecieć do kwiatka w czasie 1 sekundy? (A w czasie 0 s?)

## 6. 🏁 Wyzwanie: pszczoła ma limit prędkości

Prawdziwa pszczoła nie poleci szybciej niż około **3 m/s**. Czyli $|\vec{v}| \le 3$ m/s.

**Jaki jest najkrótszy możliwy czas lotu do kwiatka?**

Wskazówka: kierunek prędkości musi być taki sam jak kierunek przemieszczenia, więc wartości spełniają zależność $|\vec{v}| = \dfrac{|\Delta\vec{r}|}{\Delta t}$. Policz to w Pythonie, a potem sprawdź swoją odpowiedź na wykresie (z komórki wyżej).

In [ ]:
dr_potrzebne = kwiatek - r1
odleglosc = np.linalg.norm(dr_potrzebne)

print(f"Pszczoła musi pokonać {odleglosc:.2f} m w linii prostej")

# uzupełnij: ile sekund to potrwa przy prędkości 3 m/s?
# najkrotszy_czas = ...

<details>
<summary>💡 Podpowiedź</summary>

Przekształć wzór na wartość prędkości: $\Delta t = \dfrac{|\Delta\vec{r}|}{|\vec{v}|}$. W Pythonie: `najkrotszy_czas = odleglosc / 3`.
Potem wylicz `v = dr_potrzebne / najkrotszy_czas`, sprawdź `np.linalg.norm(v)` i narysuj lot.
</details>

## 7. Podsumowanie

| Pojęcie | Wzór | W Pythonie |
|---|---|---|
| Przemieszczenie | $\Delta\vec{r} = \vec{r}_2 - \vec{r}_1$ | `dr = r2 - r1` |
| Czas lotu | $\Delta t = t_2 - t_1$ | `dt = t2 - t1` |
| Prędkość (wektor) | $\vec{v} = \dfrac{\Delta\vec{r}}{\Delta t}$ | `v = dr / dt` |
| Wartość prędkości | $\lvert\vec{v}\rvert = \sqrt{v_x^2 + v_y^2 + v_z^2}$ | `np.linalg.norm(v)` |
| Położenie po czasie $\Delta t$ | $\vec{r}_2 = \vec{r}_1 + \vec{v}\,\Delta t$ | `r2 = r1 + v * dt` |

**Najważniejsze:** prędkość jest **wektorem** – ma kierunek zgodny z przemieszczeniem. Jej długość to wartość prędkości, mierzona w m/s.

**W następnej części:** co, jeśli pszczoła nie leci ze stałą prędkością, tylko przyspiesza albo hamuje? Wprowadzimy **przyspieszenie**.

---
### 🔎 Dla ciekawskich: jak działa animacja?
W kodzie pomocniczym funkcja `lot` liczy położenie pszczoły dla 41 chwil czasu między $0$ a $\Delta t$ ze wzoru $\vec{r} = \vec{r}_1 + \vec{v}\,t$ i układa z nich „klatki" animacji. Tak samo działałaby pętla w prostym programie. Spróbuj zmienić liczbę klatek `N` albo kolor wektora prędkości (`"royalblue"`).

---

[← Lekcja 1: Położenie](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_1_polozenie.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 2 z 2** (następne wkrótce)